# 코랩 YOLO11n 음식 위치 탐지 학습

음식 이미지 1,036장과 `metadata.csv`의 Bounding Box로 `food` 단일 클래스 YOLO11n 모델을 학습하고, 검증 지표와 예측 이미지를 저장합니다. **Google Drive에 프로젝트와 데이터가 있어야 합니다.** 코랩은 `C:\dev`를 직접 읽을 수 없습니다.

## 1. 준비

Google Drive에 다음 두 폴더를 업로드하거나 동기화합니다.

- `MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline`
- `MyDrive/final_1_team/data/processed/aihub_food_image_text/v2/food_description_data`

데이터 전체는 약 4.4GB이므로 Drive 여유 공간을 확보하세요.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/final_1_team')
PROJECT_ROOT = DRIVE_ROOT / 'apps/api/food-image-cleanup-pipeline'
DATASET_SOURCE_ROOT = DRIVE_ROOT / 'data/processed/aihub_food_image_text/v2/food_description_data'
assert (PROJECT_ROOT / 'scripts/prepare_yolo_food_dataset.py').is_file(), PROJECT_ROOT
assert (DATASET_SOURCE_ROOT / 'images').is_dir(), DATASET_SOURCE_ROOT
assert (DATASET_SOURCE_ROOT / 'metadata.csv').is_file(), DATASET_SOURCE_ROOT / 'metadata.csv'
%cd $PROJECT_ROOT

## 2. 안전한 학습 의존성 설치

코랩 기본 CUDA PyTorch를 교체하지 않습니다. YOLO11n 학습에 필요한 Ultralytics만 별도 폴더에 설치합니다.

In [ ]:
import os, subprocess, sys
PACKAGE_DIR = Path('/content/yolo-food-training-packages')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(PACKAGE_DIR), '--no-deps', 'ultralytics>=8.3,<9.0', 'ultralytics-thop>=2.0'], check=True)
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV['PYTHONPATH'] = str(PACKAGE_DIR) + os.pathsep + RUNTIME_ENV.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__); print("CUDA 사용 가능:", torch.cuda.is_available()); print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")'], check=True, env=RUNTIME_ENV)
subprocess.run([sys.executable, '-c', 'from ultralytics import YOLO; print("Ultralytics 확인 완료")'], check=True, env=RUNTIME_ENV)

## 3. 라벨 데이터셋 생성

원본은 수정하지 않습니다. 학습용 `data/training`에 이미지 하드링크/복사본과 YOLO 라벨을 생성합니다. 이미 생성했다면 이 셀을 건너뜁니다.

In [ ]:
import shutil
OUTPUT_ROOT = PROJECT_ROOT / 'data/training/yolo_food_detection'
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)  # 학습용 파생 데이터만 다시 생성합니다. 원본 images/는 건드리지 않습니다.
subprocess.run([sys.executable, '-m', 'scripts.prepare_yolo_food_dataset', '--source-root', str(DATASET_SOURCE_ROOT), '--output-root', str(OUTPUT_ROOT), '--copy-images'], check=True, env=RUNTIME_ENV)
print((OUTPUT_ROOT / 'dataset_audit.txt').read_text(encoding='utf-8'))

## 4. YOLO11n 학습

초기 검증은 100 epoch, 960px로 실행합니다. 실행이 중단되면 같은 이름 대신 새 이름을 사용하거나 `last.pt`를 `--weights`로 지정해 재개합니다.

In [ ]:
RUN_NAME = 'yolo11n_food_v1'
subprocess.run([sys.executable, '-m', 'scripts.train_yolo11n_food_detector', '--data', str(OUTPUT_ROOT / 'dataset.yaml'), '--epochs', '100', '--imgsz', '960', '--device', '0', '--project', 'runs/yolo_food_detector', '--name', RUN_NAME], check=True, env=RUNTIME_ENV)
BEST_WEIGHTS = PROJECT_ROOT / 'runs/yolo_food_detector' / RUN_NAME / 'weights/best.pt'
assert BEST_WEIGHTS.is_file(), BEST_WEIGHTS
print('학습 완료:', BEST_WEIGHTS)

## 5. 정량 평가

정밀도, 재현율, mAP@0.5, mAP@0.75, mAP@0.5:0.95를 `metrics.json`으로 저장합니다. 운영 적용 전에는 mAP 숫자뿐 아니라 다음 예측 이미지도 반드시 확인합니다.

In [ ]:
subprocess.run([sys.executable, '-m', 'scripts.evaluate_yolo11n_food_detector', '--weights', str(BEST_WEIGHTS), '--data', str(OUTPUT_ROOT / 'dataset.yaml'), '--imgsz', '960', '--device', '0', '--project', 'runs/yolo_food_detector_evaluation', '--name', RUN_NAME], check=True, env=RUNTIME_ENV)
METRICS_PATH = PROJECT_ROOT / 'runs/yolo_food_detector_evaluation' / RUN_NAME / 'metrics.json'
print(METRICS_PATH.read_text(encoding='utf-8'))

## 6. 검증 예측 시각화

정량 지표가 좋아도 음식 일부만 잡히거나 접시를 제외하면 배경 합성 품질이 낮아질 수 있으므로, 예측 결과를 눈으로 검토합니다.

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Image as DisplayImage
model = YOLO(str(BEST_WEIGHTS))
preview_dir = PROJECT_ROOT / 'runs/yolo_food_detector_preview'
results = model.predict(source=str(OUTPUT_ROOT / 'images/val'), imgsz=960, conf=0.25, save=True, project=str(preview_dir), name=RUN_NAME, exist_ok=True, device=0)
saved_dir = preview_dir / RUN_NAME
for image_path in list(saved_dir.glob('*'))[:12]:
    display(DisplayImage(filename=str(image_path)))
print('예측 이미지:', saved_dir)